# Ứng dụng 2 · Các mẫu DNA quanh ranh giới exon–intron

Bộ dữ liệu có 3.190 đoạn DNA của linh trưởng, mỗi đoạn dài 60 nucleotide. Các đoạn được chia thành ba nhóm **EI** (exon → intron), **IE** (intron → exon) và **N** (đối chứng).

Chọn runtime **Python 3, CPU** trên Colab rồi chạy các ô từ trên xuống. Notebook sẽ chuẩn bị Julia, đọc dữ liệu UCI và so sánh các mẫu giữa ba nhóm.

Nguồn: [UCI Molecular Biology (Splice-junction Gene Sequences), 1991](https://doi.org/10.24432/C5M888) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

In [ ]:
# @title Chuẩn bị Julia và thư mục làm việc
from pathlib import Path
import csv
import hashlib
import platform
import shutil
import subprocess
import tempfile
import urllib.request

base = Path("/content") if Path("/content").is_dir() else Path.cwd()
cache = base / "splice-junction-cache"
cache.mkdir(exist_ok=True)
project = Path(tempfile.mkdtemp(prefix="splice-junction-", dir=base))

def verified_download(url, dest, expected_sha256):
    def digest(path):
        h = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    if dest.is_file() and digest(dest) == expected_sha256:
        return dest
    with tempfile.NamedTemporaryFile(dir=dest.parent, delete=False) as temp:
        staging = Path(temp.name)
    try:
        urllib.request.urlretrieve(url, staging)
        if digest(staging) != expected_sha256:
            raise ValueError(f"Tệp tải về không khớp SHA-256: {dest.name}")
        staging.replace(dest)
    finally:
        staging.unlink(missing_ok=True)
    return dest

julia = shutil.which("julia")
if not julia:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("Hãy chọn Python 3 trên Colab hoặc thêm Julia 1.9+ vào PATH.")
    julia = str(cache / "julia-1.11.4" / "bin" / "julia")
    if not Path(julia).is_file():
        archive = verified_download(
            "https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz",
            cache / "julia-1.11.4-linux-x86_64.tar.gz",
            "fb3d3c5fccef82158a70677c0044ac5ae40410eceb0604cdc8e643eeff21df8d",
        )
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(cache)], check=True)
print(subprocess.check_output([julia, "--version"], text=True).strip())
print("Thư mục kết quả của phiên:", project)

In [ ]:
# @title Mã nguồn Julia của nhóm (mở ô để xem)
# Phiên bản mã nguồn trong repository: e48110f6978f226a80329dbcfeef07e1c519c530
source_files = {
    "Project.toml": r'''name = "FrequentItemsetMining"
uuid = "284f3ddb-453b-4c46-9ce6-1e85fd28ca9d"
version = "0.1.0"

[deps]
Dates = "ade2ca70-3891-5945-98fb-dc099432e06a"
DelimitedFiles = "8bb1440f-4735-579b-a4ab-409b98df4dab"
Printf = "de0858da-6303-5e67-8744-51eddeeeb8d7"
SHA = "ea8e919c-243c-51af-8825-aaa63cd721ce"
TOML = "fa267f1f-6049-4f14-aa54-33bafae1ed76"
UUIDs = "cf7118a7-6976-5b1a-9a39-7adc72f591a4"

[compat]
julia = "1.9"

[extras]
Test = "8dfed614-e22c-5e08-85e1-65c5234f0b40"
Random = "9a3f8284-a2c9-5f02-9a11-845980a1fd5c"

[targets]
test = ["Test", "Random"]
''',
    "src/apriori.jl": r'''"""Statistics for a level actually counted (Figure 1 of the paper)."""
struct LevelStats
    size::Int
    candidates::Int
    frequent::Int
end

"""Measured monotonic duration for one mining phase; level totals are nested."""
struct PhaseTiming
    phase::Symbol
    size::Int
    elapsed_ns::UInt64
    candidates::Int
    frequent::Int
end

"""All frequent itemsets and integer transaction support counts."""
struct MiningResult
    # Variable-length keys occur only at the result boundary. Counting uses
    # concrete NTuple{K,Int} vectors and Int counters inside each level.
    itemsets::Dict{Tuple{Vararg{Int}},Int}
    transaction_count::Int
    min_support_count::Int
    levels::Vector{LevelStats}
    timings::Vector{PhaseTiming}
    mining_ns::UInt64
end

MiningResult(itemsets, transaction_count, min_support_count, levels) =
    MiningResult(itemsets, transaction_count, min_support_count, levels, PhaseTiming[], UInt64(0))

record_phase!(::Nothing, phase, size, elapsed_ns, candidates, frequent) = nothing
function record_phase!(timings::Vector{PhaseTiming}, phase, size, elapsed_ns, candidates, frequent)
    push!(timings, PhaseTiming(phase, size, elapsed_ns, candidates, frequent))
    return nothing
end

function normalize_transactions(transactions)
    database = Vector{Int}[]
    for (row_number, row) in enumerate(transactions)
        items = Int[]
        for item in row
            if item isa Bool || !(item isa Integer) || item < 0 || item > typemax(Int)
                throw(ArgumentError("transaction $row_number: items must be nonnegative Int values"))
            end
            push!(items, Int(item))
        end
        sort!(items)
        unique!(items)
        push!(database, items)
    end
    return database
end

"""Parse decimal/scientific notation without an intermediate binary float."""
function decimal_ratio(text::AbstractString)
    parsed = match(r"^([+-]?)(\d*)(?:\.(\d*))?(?:[eE]([+-]?\d+))?$", strip(text))
    parsed === nothing && throw(ArgumentError("min_support must be a finite decimal ratio"))
    sign, whole, fraction, exponent = parsed.captures
    fraction = isnothing(fraction) ? "" : fraction
    isempty(whole) && isempty(fraction) && throw(ArgumentError("min_support has no digits"))
    power = isnothing(exponent) ? 0 : tryparse(Int, exponent)
    (isnothing(power) || abs(big(power)) > 10000) &&
        throw(ArgumentError("min_support exponent is outside the supported range"))
    numerator = parse(BigInt, (isempty(whole) ? "0" : whole) * fraction)
    sign == "-" && (numerator = -numerator)
    scale = length(fraction) - power
    return scale >= 0 ? numerator // big(10)^scale : (numerator * big(10)^(-scale)) // big(1)
end

function support_ratio(value::Rational)
    return big(numerator(value)) // big(denominator(value))
end
support_ratio(value::AbstractString) = decimal_ratio(value)
support_ratio(value::Integer) = big(value) // big(1)
function support_ratio(value::AbstractFloat)
    isfinite(value) || throw(ArgumentError("min_support must be finite"))
    return decimal_ratio(string(value))
end
support_ratio(value) = throw(ArgumentError("min_support must be a decimal string, Integer, Rational or AbstractFloat"))

function support_threshold(n::Int, min_support, min_count)
    isnothing(min_support) == isnothing(min_count) &&
        throw(ArgumentError("provide exactly one of min_support or min_count"))
    if !isnothing(min_count)
        if min_count isa Bool || !(min_count isa Integer) || !(1 <= min_count <= typemax(Int))
            throw(ArgumentError("min_count must be an integer in 1:typemax(Int)"))
        end
        return Int(min_count)
    end
    min_support isa Bool && throw(ArgumentError("min_support cannot be Bool"))
    ratio = support_ratio(min_support)
    0 < ratio <= 1 || throw(ArgumentError("min_support must be in (0, 1]"))
    return Int(cld(numerator(ratio) * n, denominator(ratio)))
end

"""§2.1.1: prefix join followed by membership pruning of every K-subset."""
function apriori_gen(previous::Vector{NTuple{K,Int}}, trace::AbstractAprioriTrace=NoTrace()) where K
    K >= 1 || throw(ArgumentError("previous itemsets must not be empty"))
    ordered = sort!(unique(previous))
    known = Set(ordered)
    candidates = NTuple{K+1,Int}[]
    for i in eachindex(ordered)
        left = ordered[i]
        # Index iteration avoids copying every remaining tail of the list.
        for j in (i + 1):length(ordered)
            right = ordered[j]
            all(left[p] == right[p] for p in 1:(K - 1)) || break
            candidate = (left..., right[K])
            record!(trace, :join, left, right, candidate)
            valid = true
            for removed in 1:(K + 1)
                subset = ntuple(p -> candidate[p + (p >= removed)], Val(K))
                if !(subset in known)
                    record!(trace, :prune, candidate, subset, K)
                    valid = false
                    break
                end
            end
            if valid
                push!(candidates, candidate)
                record!(trace, :accepted, candidate)
            end
        end
    end
    return candidates
end


"""
    foreach_candidate_batch(callback, previous, batch_size, trace)

Generate Apriori candidates with the same prefix-join and all-subsets pruning
rules as `apriori_gen`, but deliver bounded batches to `callback`. The callback
must finish counting/filtering a batch before returning; candidate tuples are
not retained by this function after the callback.
"""
function foreach_candidate_batch(callback::F, previous::Vector{NTuple{K,Int}},
                                 batch_size::Int, trace::AbstractAprioriTrace) where {K,F}
    K >= 1 || throw(ArgumentError("previous itemsets must not be empty"))
    batch_size >= 1 || throw(ArgumentError("candidate_batch_size must be >= 1"))
    generation_started = time_ns()
    ordered = issorted(previous) && allunique(previous) ? previous : sort!(unique(previous))
    known = Set(ordered)
    batch = NTuple{K+1,Int}[]
    total = 0
    generation_ns = UInt64(0)

    for i in eachindex(ordered)
        left = ordered[i]
        for j in (i + 1):length(ordered)
            right = ordered[j]
            all(left[p] == right[p] for p in 1:(K - 1)) || break
            candidate = (left..., right[K])
            record!(trace, :join, left, right, candidate)
            valid = true
            for removed in 1:(K + 1)
                subset = ntuple(p -> candidate[p + (p >= removed)], Val(K))
                if !(subset in known)
                    record!(trace, :prune, candidate, subset, K)
                    valid = false
                    break
                end
            end
            if valid
                push!(batch, candidate)
                total += 1
                record!(trace, :accepted, candidate)
                if length(batch) == batch_size
                    generation_ns += time_ns() - generation_started
                    callback(batch)
                    empty!(batch)
                    generation_started = time_ns()
                end
            end
        end
    end

    generation_ns += time_ns() - generation_started
    isempty(batch) || callback(batch)
    return (; total, generation_ns)
end

function count_candidates(candidates::Vector{NTuple{K,Int}}, database::Vector{Vector{Int}},
                          ::Val{:scan}, trace::AbstractAprioriTrace=NoTrace(),
                          timings::Union{Nothing,Vector{PhaseTiming}}=nothing) where K
    # Scan has no reusable counter structure to build.
    record_phase!(timings, :build_counter, K, UInt64(0), length(candidates), 0)
    started = time_ns()
    counts = zeros(Int, length(candidates))
    for (tid, transaction) in enumerate(database)
        record!(trace, :transaction, K, tid, transaction)
        present = Set(transaction)
        for id in eachindex(candidates)
            if all(item in present for item in candidates[id])
                counts[id] += 1
                record!(trace, :increment, tid, candidates[id], counts[id])
            end
        end
    end
    record_phase!(timings, :count_support, K, time_ns() - started, length(candidates), 0)
    return counts
end

function count_candidates(candidates::Vector{NTuple{K,Int}}, database::Vector{Vector{Int}},
                          ::Val{:hash_tree}, trace::AbstractAprioriTrace=NoTrace(),
                          timings::Union{Nothing,Vector{PhaseTiming}}=nothing) where K
    started = time_ns()
    tree = HashTree(candidates)
    workspace = HashQueryWorkspace(tree)
    counts = zeros(Int, length(candidates))
    record_phase!(timings, :build_counter, K, time_ns() - started, length(candidates), 0)
    started = time_ns()
    for (tid, transaction) in enumerate(database)
        record!(trace, :transaction, K, tid, transaction)
        for id in ordered_matches!(trace, matching_ids!(workspace, tree, transaction))
            counts[id] += 1
            record!(trace, :increment, tid, candidates[id], counts[id])
        end
    end
    record_phase!(timings, :count_support, K, time_ns() - started, length(candidates), 0)
    return counts
end

function next_level(previous::Vector{NTuple{K,Int}}, database, threshold, mode,
                    trace::AbstractAprioriTrace=NoTrace(),
                    timings::Union{Nothing,Vector{PhaseTiming}}=nothing;
                    candidate_batch_size::Int=10_000) where K
    record!(trace, :level, K+1)
    frequent = NTuple{K+1,Int}[]
    supports = Int[]
    generation_phase_index = Ref(0)

    candidate_stats = foreach_candidate_batch(previous, candidate_batch_size, trace) do candidates
        if !isnothing(timings) && iszero(generation_phase_index[])
            push!(timings, PhaseTiming(:generate_candidates, K+1, UInt64(0), 0, 0))
            generation_phase_index[] = length(timings)
        end
        counts = count_candidates(candidates, database, mode, trace, timings)
        filter_started = time_ns()
        frequent_in_batch = 0
        for id in eachindex(candidates)
            record!(trace, :decision, candidates[id], counts[id], threshold)
            if counts[id] >= threshold
                push!(frequent, candidates[id])
                push!(supports, counts[id])
                frequent_in_batch += 1
            end
        end
        record_phase!(timings, :filter, K+1, time_ns() - filter_started,
                      length(candidates), frequent_in_batch)
    end

    candidate_total = candidate_stats.total
    explain_frequent!(trace, K+1, frequent, supports)
    if !isnothing(timings)
        if iszero(generation_phase_index[])
            record_phase!(timings, :generate_candidates, K+1,
                          candidate_stats.generation_ns, candidate_total, 0)
        else
            timings[generation_phase_index[]] =
                PhaseTiming(:generate_candidates, K+1, candidate_stats.generation_ns,
                            candidate_total, 0)
        end
    end
    record!(trace, :candidates, K+1, candidate_total)
    return frequent, supports, candidate_total
end

counter_mode(counter) = counter in (:scan, "scan") ? Val(:scan) :
                        counter in (:hash_tree, "hash-tree") ? Val(:hash_tree) :
                        throw(ArgumentError("counter must be :scan or :hash_tree"))

"""
    mine(transactions; min_support=nothing, min_count=nothing, counter=:hash_tree, explain=nothing)

Plain Apriori (Figure 1): join/prune, scan original transactions, filter L_k.
Pass exactly one support threshold. Empty API rows still contribute to N.
All data is materialized in RAM; §2.1.3 external-memory handling is deferred.
"""
function mine(transactions; min_support=nothing, min_count=nothing, counter=:hash_tree,
              explain=nothing, candidate_batch_size::Int=10_000)
    mining_started = time_ns()
    candidate_batch_size >= 1 ||
        throw(ArgumentError("candidate_batch_size must be >= 1"))
    mode = counter_mode(counter)
    trace = isnothing(explain) ? NoTrace() : explain
    trace isa AbstractAprioriTrace || throw(ArgumentError("explain must be ExplainTrace or nothing"))
    timings = PhaseTiming[]
    started = time_ns()
    database = normalize_transactions(transactions)
    record_phase!(timings, :normalize, 0, time_ns() - started, 0, 0)
    started = time_ns()
    threshold = support_threshold(length(database), min_support, min_count)
    record_phase!(timings, :threshold, 0, time_ns() - started, 0, 0)
    answer = Dict{Tuple{Vararg{Int}},Int}()
    levels = LevelStats[]
    mode_name = mode isa Val{:scan} ? "scan" : "hash-tree"
    record!(trace, :begin, length(database), threshold, mode_name)
    explain_input!(trace, database, min_support, threshold)
    if isempty(database)
        record!(trace, :stop, "CSDL rỗng.")
        finish_explanation!(trace, 0)
        return MiningResult(answer, 0, threshold, levels, timings, time_ns() - mining_started)
    end

    # Figure 1, line 1: count singletons, then select L_1.
    level_started = time_ns()
    started = time_ns()
    singletons = Dict{Int,Int}()
    record!(trace, :level, 1)
    for (tid, transaction) in enumerate(database)
        record!(trace, :transaction, 1, tid, transaction)
        for item in transaction
            singletons[item] = get(singletons, item, 0) + 1
            record!(trace, :increment, tid, (item,), singletons[item])
        end
    end
    record_phase!(timings, :singleton_count, 1, time_ns() - started, length(singletons), 0)
    started = time_ns()
    explain_singletons!(trace, singletons, threshold)
    previous = NTuple{1,Int}[(item,) for (item, count) in singletons if count >= threshold]
    sort!(previous)
    record_phase!(timings, :filter, 1, time_ns() - started, length(singletons), length(previous))
    for itemset in previous
        answer[itemset] = singletons[itemset[1]]
    end
    push!(levels, LevelStats(1, length(singletons), length(previous)))
    # Support values are already in the result; no extra collection when off.
    explain_frequent!(trace, 1, previous, (singletons[itemset[1]] for itemset in previous))
    isempty(previous) && record!(trace, :stop, "L_1 rỗng, không có item frequent.")
    record_phase!(timings, :level_total, 1, time_ns() - level_started, length(singletons), length(previous))

    # next_level is a function barrier: its loops specialize for each K.
    size = 2
    while !isempty(previous)
        level_started = time_ns()
        current, supports, candidate_count = next_level(previous, database, threshold, mode, trace, timings; candidate_batch_size)
        if iszero(candidate_count)
            record!(trace, :stop, "C_$size rỗng, không còn ứng viên để đếm.")
            record_phase!(timings, :level_total, size, time_ns() - level_started, 0, 0)
            break
        end
        push!(levels, LevelStats(size, candidate_count, length(current)))
        for id in eachindex(current)
            answer[current[id]] = supports[id]
        end
        previous = current
        isempty(previous) && record!(trace, :stop, "L_$size rỗng, mọi ứng viên đều dưới ngưỡng.")
        record_phase!(timings, :level_total, size, time_ns() - level_started, candidate_count, length(current))
        size += 1
    end
    finish_explanation!(trace, length(answer))
    return MiningResult(answer, length(database), threshold, levels, timings, time_ns() - mining_started)
end
''',
    "src/cli.jl": r'''const CLI_HELP = """
Plain Apriori from the original paper (Julia, no FIM library).
Usage: julia --project=. bin/apriori.jl --input INPUT [--output OUTPUT]
       (--minsup RATIO | --min-count COUNT) [--counter scan|hash-tree] [--trace]
       [--explain | --explain-file PATH] [--explain-limit N]
       [--report-dir PATH] [--warmup N] [--quiet]

--minsup     Minimum support ratio in (0, 1], e.g. 0.5
--min-count  Minimum absolute transaction count >= 1
--counter    scan or hash-tree (default: hash-tree)
--trace      Print per-level C_k / L_k statistics to stderr
--explain    Explain the actual Apriori steps in Vietnamese on stderr
--explain-file PATH  Write the explanation as UTF-8 to a separate file
--explain-limit N    Maximum detail lines (positive integer; default: 5000)
--report-dir PATH   New directory for this run's immutable report bundle
--warmup N          Untraced mining runs before the measured run (default: 0)
--quiet             Suppress success messages; reports are still written
--help       Show this help
Reports default to output/runs/<dataset>-N/ or <output-parent>/runs/<dataset>-N/.
Without --output, itemsets.txt is written directly inside the report directory.
Output parent must exist unless it is inside the new report directory.
"""

function parse_cli(args)
    values = Dict{String,String}()
    trace = false
    explain = false
    quiet = false
    index = 1
    while index <= length(args)
        option = args[index]
        if option == "--trace"
            trace && throw(ArgumentError("duplicate --trace"))
            trace = true
            index += 1
        elseif option == "--explain"
            explain && throw(ArgumentError("duplicate --explain"))
            explain = true
            index += 1
        elseif option == "--quiet"
            quiet && throw(ArgumentError("duplicate --quiet"))
            quiet = true
            index += 1
        elseif option in ("--input", "--output", "--minsup", "--min-count", "--counter",
                          "--explain-file", "--explain-limit", "--report-dir", "--warmup")
            haskey(values, option) && throw(ArgumentError("duplicate $option"))
            index < length(args) || throw(ArgumentError("missing value for $option"))
            values[option] = args[index + 1]
            index += 2
        else
            throw(ArgumentError("unknown option: $option"))
        end
    end
    haskey(values, "--input") || throw(ArgumentError("required option: --input"))
    haskey(values, "--minsup") == haskey(values, "--min-count") &&
        throw(ArgumentError("provide exactly one of --minsup or --min-count"))
    return values, trace, explain, quiet
end

function cli_same_path(left, right)
    a, b = cli_resolved_path(left), cli_resolved_path(right)
    Sys.iswindows() && ((a, b) = (lowercase(a), lowercase(b)))
    return a == b || (ispath(left) && ispath(right) && samefile(left, right))
end

function cli_inside(path, directory)
    candidate, parent = cli_resolved_path(path), cli_resolved_path(directory)
    Sys.iswindows() && ((candidate, parent) = (lowercase(candidate), lowercase(parent)))
    relative = relpath(candidate, parent)
    return relative != "." && !isabspath(relative) &&
           first(splitpath(relative)) != ".."
end

function check_cli_report_paths(report_dir, input, output, explanation_path)
    destinations = isnothing(explanation_path) ? [output] : [output, explanation_path]
    for destination in destinations
        islink(destination) && !ispath(destination) &&
            throw(ArgumentError("destination is a dangling symlink: $destination"))
        cli_same_path(destination, report_dir) &&
            throw(ArgumentError("destination must be a file, not the report directory"))
        cli_inside(report_dir,destination) &&
            throw(ArgumentError("report directory cannot be inside a destination file path"))
        isdir(destination) && throw(ArgumentError("destination is a directory: $destination"))
        if !isdir(dirname(abspath(destination))) && !cli_inside(destination, report_dir)
            throw(ArgumentError("destination parent directory must already exist: $destination"))
        end
    end
    paths = isnothing(explanation_path) ? [input, output] : [input, output, explanation_path]
    for name in ("report.txt", "timings.csv", "metadata.toml", "itemsets.txt", "trace.txt")
        artifact = joinpath(report_dir, name)
        for (index, path) in enumerate(paths)
            allowed = name == "itemsets.txt" && index == 2 ||
                      name == "trace.txt" && index == 3
            if (cli_same_path(path, artifact) && !allowed) || cli_inside(path, artifact)
                throw(ArgumentError("path conflicts with reserved report artifact $name"))
            end
        end
    end
    # Check ancestors without creating them, including a file where a directory
    # would be needed. The final directory itself is claimed with mkdir below.
    ancestor = dirname(abspath(report_dir))
    while !ispath(ancestor)
        ancestor = dirname(ancestor)
    end
    isdir(ancestor) || throw(ArgumentError("report parent is not a directory: $ancestor"))
    return nothing
end

# Resolve existing ancestors too, so an output that does not yet exist is
# still compared correctly when its parent is reached through a symlink.
function cli_resolved_path(path)
    absolute = normpath(abspath(path))
    ispath(absolute) && return realpath(absolute)
    parent = dirname(absolute)
    parent == absolute && return absolute
    return joinpath(cli_resolved_path(parent), basename(absolute))
end

function check_cli_paths(paths)
    resolved = cli_resolved_path.(paths)
    Sys.iswindows() && (resolved = lowercase.(resolved))
    for left in eachindex(paths), right in (left + 1):length(paths)
        same_path = resolved[left] == resolved[right]
        same_existing = ispath(paths[left]) && ispath(paths[right]) &&
                        samefile(paths[left], paths[right])
        (same_path || same_existing) &&
            throw(ArgumentError("input, output and explanation must not refer to the same file"))
        (cli_inside(paths[left], paths[right]) || cli_inside(paths[right], paths[left])) &&
            throw(ArgumentError("a file destination cannot also be another path's parent directory"))
    end
    return nothing
end

function main(args=ARGS; out=stdout, err=stderr)
    if args == ["--help"]
        print(out, CLI_HELP)
        return 0
    end
    try
        values, trace, explain, quiet = parse_cli(args)
        cli_start = time_ns()
        started = Dates.now(Dates.UTC)
        run_id = Dates.format(started, dateformat"yyyymmddTHHMMSSsss") * "_" * string(uuid4())
        input = abspath(values["--input"])
        requested_output = haskey(values, "--output") ? abspath(values["--output"]) : nothing
        explanation_path = haskey(values, "--explain-file") ? abspath(values["--explain-file"]) : nothing
        output_for(dir) = something(requested_output, joinpath(dir, "itemsets.txt"))
        function validate_paths(dir)
            output = output_for(dir)
            paths = [input, output]
            !isnothing(explanation_path) && push!(paths, explanation_path)
            check_cli_paths(paths)
            check_cli_report_paths(dir, input, output, explanation_path)
        end
        min_count = nothing
        if haskey(values, "--min-count")
            min_count = tryparse(Int, values["--min-count"])
            isnothing(min_count) && throw(ArgumentError("--min-count must be an integer"))
        end
        limit = tryparse(Int, get(values, "--explain-limit", "5000"))
        isnothing(limit) && throw(ArgumentError("--explain-limit must be a positive integer"))
        explanation = ExplainTrace(err; limit)
        warmup = tryparse(Int, get(values, "--warmup", "0"))
        (isnothing(warmup) || warmup < 0) &&
            throw(ArgumentError("--warmup must be a nonnegative integer"))
        min_support = get(values, "--minsup", nothing)
        counter = get(values, "--counter", "hash-tree")
        # Validate all input and mining options before opening either destination.
        read_start = time_ns()
        # Parse and fingerprint the same immutable byte snapshot. A file
        # changed after this read cannot be relabelled as the mined input.
        input_bytes=read(input)
        input_sha256=bytes2hex(sha256(input_bytes))
        rows = read_spmf(IOBuffer(input_bytes))
        read_ns = time_ns() - read_start
        support_threshold(length(rows), min_support, min_count)
        counter_mode(counter)
        source_sha256=implementation_fingerprint()
        # All validation has finished before any destination is created/opened.
        if haskey(values, "--report-dir")
            report_dir = abspath(values["--report-dir"])
            (ispath(report_dir) || islink(report_dir)) &&
                throw(ArgumentError("report directory must not already exist: $report_dir"))
            validate_paths(report_dir)
            mkpath(dirname(report_dir))
            mkdir(report_dir)
        else
            parent = isnothing(requested_output) ? abspath("output") : dirname(requested_output)
            stem = splitext(basename(input))[1]
            report_dir = numbered_run_directory(joinpath(parent, "runs"), stem; validate=validate_paths)
        end
        output = output_for(report_dir)
        for destination in (output, explanation_path)
            if !isnothing(destination) && cli_inside(destination, report_dir)
                mkpath(dirname(destination))
            end
        end
        warmup_ns = UInt64(0)
        if warmup > 0
            warmup_start = time_ns()
            for _ in 1:warmup
                mine(rows; min_support, min_count, counter)
            end
            warmup_ns = time_ns() - warmup_start
        end
        result = if !isnothing(explanation_path)
            open(explanation_path, "w") do stream
                mine(rows; min_support, min_count, counter,
                     explain=ExplainTrace(stream; limit))
            end
        else
            mine(rows; min_support, min_count, counter,
                 explain=explain ? explanation : nothing)
        end
        write_start = time_ns()
        write_spmf(result.itemsets, output)
        write_ns = time_ns() - write_start
        if trace
            println(err, "transactions=", result.transaction_count, " min_count=",
                    result.min_support_count, " itemsets=", length(result.itemsets))
            for level in result.levels
                println(err, "k=", level.size, " candidates=", level.candidates,
                        " frequent=", level.frequent)
            end
        end
        cli_ns = time_ns() - cli_start
        metadata = Dict{String,Any}(
            "run_id" => run_id,
            "started_utc" => Dates.format(started, dateformat"yyyy-mm-ddTHH:MM:SS.sssZ"),
            "input_path" => input, "minsup_argument" => get(values, "--minsup", ""),
            "input_sha256" => input_sha256, "input_bytes" => length(input_bytes),
            "input_hash_scope" => "exact_bytes_parsed", "implementation_sha256" => source_sha256,
            "counter" => counter, "warmup_runs" => warmup,
            "explain_enabled" => explain || !isnothing(explanation_path), "explain_limit" => limit)
        write_run_report(report_dir, result; metadata, output_path=output, explanation_path,
                         read_ns, write_ns, warmup_ns, cli_ns)
        if !quiet
            println(out, "Report: ", report_dir)
            println(out, "MiningMs: ", result.mining_ns / 1_000_000)
        end
        return 0
    catch exc
        if exc isa ArgumentError || exc isa SystemError || exc isa Base.IOError
            print(err, "apriori: error: ")
            showerror(err, exc)
            println(err)
            return 2
        end
        rethrow()
    end
end
''',
    "src/explain.jl": r'''abstract type AbstractAprioriTrace end
struct NoTrace <: AbstractAprioriTrace end

"""Stream real Apriori events, limiting detail lines without limiting mining.

    ExplainTrace(io=stderr; limit=5000)

Create a fresh trace for each mine() call. The final summary is always printed,
even when the detail limit was reached. No event history is kept in memory.
"""
mutable struct ExplainTrace{T<:IO} <: AbstractAprioriTrace
    io::T
    limit::Int
    emitted::Int
    truncated::Bool
end

function ExplainTrace(io::IO=stderr; limit=5000)
    if limit isa Bool || !(limit isa Integer) || !(1 <= limit <= typemax(Int))
        throw(ArgumentError("explain limit must be a positive integer fitting Int"))
    end
    return ExplainTrace(io, Int(limit), 0, false)
end

item_text(items) = "{" * join(items, ',') * "}"
record!(::NoTrace, event, args...) = nothing

function record!(trace::ExplainTrace, event::Symbol, args...)
    if trace.emitted >= trace.limit
        if !trace.truncated
            println(trace.io, "[Log đã đạt giới hạn ", trace.limit,
                    " dòng chi tiết; thuật toán vẫn tiếp tục tính đầy đủ.]")
            trace.truncated = true
        end
        return nothing
    end
    text = if event === :begin
        n, threshold, mode = args
        "BẮT ĐẦU Apriori: N=$n; min_count=$threshold; counter=$mode"
    elseif event === :threshold
        ratio, n, threshold = args
        "Ngưỡng support: ceil($ratio × $n) = $threshold"
    elseif event === :input
        tid, row = args
        "Dữ liệu chuẩn hóa: T$tid = $(item_text(row))"
    elseif event === :level
        "MỨC k=$(args[1])"
    elseif event === :transaction
        k, tid, row = args
        "Quét T$tid = $(item_text(row)) (k=$k)"
    elseif event === :increment
        tid, candidate, count = args
        "  T$tid chứa $(item_text(candidate)): count $(count-1) -> $count"
    elseif event === :join
        left, right, candidate = args
        "JOIN $(item_text(left)) + $(item_text(right)) -> $(item_text(candidate))"
    elseif event === :prune
        candidate, subset, k = args
        "PRUNE $(item_text(candidate)): thiếu tập con $(item_text(subset)) trong L_$k -> loại"
    elseif event === :accepted
        "Ứng viên $(item_text(args[1])): qua prune (mọi tập con đều frequent)"
    elseif event === :candidates
        "C_$(args[1]) sau prune: $(args[2]) ứng viên"
    elseif event === :decision
        candidate, count, threshold = args
        count >= threshold ? "GIỮ $(item_text(candidate)): support=$count >= $threshold" :
                             "LOẠI $(item_text(candidate)): support=$count < $threshold"
    elseif event === :frequent
        "L_$(args[1]): $(args[2]) tập phổ biến"
    elseif event === :frequent_item
        "  $(item_text(args[1])) #SUP: $(args[2])"
    elseif event === :stop
        "DỪNG: $(args[1])"
    else
        throw(ArgumentError("unknown Apriori trace event: $event"))
    end
    println(trace.io, text)
    trace.emitted += 1
    return nothing
end

explain_input!(::NoTrace, database, ratio, threshold) = nothing
function explain_input!(trace::ExplainTrace, database, ratio, threshold)
    !isnothing(ratio) && record!(trace, :threshold, ratio, length(database), threshold)
    for (tid, row) in enumerate(database)
        record!(trace, :input, tid, row)
    end
end

explain_singletons!(::NoTrace, counts, threshold) = nothing
function explain_singletons!(trace::ExplainTrace, counts, threshold)
    for item in sort!(collect(keys(counts)))
        record!(trace, :decision, (item,), counts[item], threshold)
    end
end

explain_frequent!(::NoTrace, k, candidates, counts) = nothing
function explain_frequent!(trace::ExplainTrace, k, candidates, counts)
    record!(trace, :frequent, k, length(candidates))
    for (candidate, count) in zip(candidates, counts)
        record!(trace, :frequent_item, candidate, count)
    end
end

ordered_matches!(::NoTrace, ids) = ids
ordered_matches!(::ExplainTrace, ids) = sort!(ids)

finish_explanation!(::NoTrace, total) = nothing
function finish_explanation!(trace::ExplainTrace, total)
    println(trace.io, "KẾT THÚC: ", total,
            " frequent itemsets; support là số giao dịch chứa itemset.")
end
''',
    "src/FrequentItemsetMining.jl": r'''module FrequentItemsetMining

using Dates
using SHA
using TOML
using UUIDs

export mine, mine_bitset, mine_streaming, apriori_gen, MiningResult, LevelStats, read_spmf, write_spmf, main
export HashTree, HashQueryWorkspace, matching_ids, matching_ids!
export ExplainTrace
export PhaseTiming, ExternalMiningSummary, mine_external, mine_bitset_external, mine_streaming_external

include("explain.jl")
include("hash_tree.jl")
include("apriori.jl")
include("io.jl")
include("apriori_optimized.jl")
include("apriori_external.jl")
include("reports.jl")
include("run_directories.jl")
include("cli.jl")

end
''',
    "src/hash_tree.jl": r'''"""A hash-tree node: candidate IDs in a leaf, bucket-to-node indices otherwise."""
mutable struct HashTreeNode
    candidate_ids::Vector{Int}
    children::Dict{Int,Int}
end

"""A same-width candidate hash tree whose IDs are input candidate positions."""
struct HashTree{K}
    candidates::Vector{NTuple{K,Int}}
    compressed_candidates::Vector{NTuple{K,Int}}
    nodes::Vector{HashTreeNode}
    item_positions::Dict{Int,Int}
    bucket_count::Int
    leaf_capacity::Int
end

function _hash_positive_int(value, name)
    if !(value isa Integer) || value isa Bool || value < 1 || value > typemax(Int)
        throw(ArgumentError("$name must be a positive integer fitting Int"))
    end
    return Int(value)
end

function _hash_candidate_item(item)
    if !(item isa Integer) || item isa Bool || item < 0 || item > typemax(Int)
        throw(ArgumentError("candidate items must be nonnegative integers fitting Int"))
    end
    return Int(item)
end

function HashTree(candidates::AbstractVector; bucket_count=17, leaf_capacity=8)
    buckets = _hash_positive_int(bucket_count, "bucket_count")
    capacity = _hash_positive_int(leaf_capacity, "leaf_capacity")
    if isempty(candidates)
        candidate_type = eltype(candidates)
        if !(candidate_type <: Tuple) || !isconcretetype(candidate_type)
            throw(ArgumentError("empty candidates must have a concrete tuple element type"))
        end
        width = fieldcount(candidate_type)
    else
        first(candidates) isa Tuple || throw(ArgumentError("candidates must be tuples"))
        width = length(first(candidates))
    end
    width > 0 || throw(ArgumentError("candidates must be nonempty tuples"))
    canonical = NTuple{width,Int}[]
    sizehint!(canonical, length(candidates))
    for candidate in candidates
        candidate isa Tuple && length(candidate) == width ||
            throw(ArgumentError("candidates must be tuples of the same width"))
        converted = ntuple(index -> _hash_candidate_item(candidate[index]), width)
        for index in 2:width
            converted[index - 1] < converted[index] ||
                throw(ArgumentError("candidate items must be strictly increasing"))
        end
        push!(canonical, converted)
    end
    return _build_hash_tree(canonical, buckets, capacity)
end

function _build_hash_tree(candidates::Vector{NTuple{K,Int}}, buckets::Int,
                          capacity::Int) where {K}
    positions = Dict{Int,Int}()
    for candidate in candidates, item in candidate
        if !haskey(positions, item)
            positions[item] = length(positions) + 1
        end
    end
    compressed = NTuple{K,Int}[ntuple(i -> positions[candidate[i]], K)
                               for candidate in candidates]
    nodes = HashTreeNode[HashTreeNode(collect(eachindex(candidates)), Dict{Int,Int}())]
    tree = HashTree{K}(candidates, compressed, nodes, positions, buckets, capacity)
    _split_hash_leaf!(tree, 1, 1)
    return tree
end

function _split_hash_leaf!(tree::HashTree{K}, node_id::Int, depth::Int) where {K}
    node = tree.nodes[node_id]
    if length(node.candidate_ids) <= tree.leaf_capacity || depth > K
        return
    end
    for candidate_id in node.candidate_ids
        bucket = mod(tree.candidates[candidate_id][depth], tree.bucket_count)
        child_id = get(node.children, bucket, 0)
        if child_id == 0
            push!(tree.nodes, HashTreeNode(Int[], Dict{Int,Int}()))
            child_id = length(tree.nodes)
            node.children[bucket] = child_id
        end
        push!(tree.nodes[child_id].candidate_ids, candidate_id)
    end
    empty!(node.candidate_ids)
    for child_id in values(node.children)
        _split_hash_leaf!(tree, child_id, depth + 1)
    end
    return
end

"""Reusable query buffers; results remain valid only until the next query."""
struct HashQueryWorkspace
    membership::BitVector
    touched_items::Vector{Int}
    visited_leaves::BitVector
    touched_leaves::Vector{Int}
    result_ids::Vector{Int}
end

HashQueryWorkspace(tree::HashTree) = HashQueryWorkspace(
    falses(length(tree.item_positions)), Int[], falses(length(tree.nodes)), Int[], Int[])

function _query_hash_node!(workspace::HashQueryWorkspace, tree::HashTree{K},
                           node_id::Int, transaction::Vector{Int}, start::Int,
                           depth::Int) where {K}
    node = tree.nodes[node_id]
    if isempty(node.children)
        workspace.visited_leaves[node_id] && return
        workspace.visited_leaves[node_id] = true
        push!(workspace.touched_leaves, node_id)
        for candidate_id in node.candidate_ids
            candidate = tree.compressed_candidates[candidate_id]
            if all(index -> workspace.membership[index], candidate)
                push!(workspace.result_ids, candidate_id)
            end
        end
        return
    end
    # A node may be reached through several transaction prefixes. Only leaves
    # can be skipped globally, after their complete candidates were checked.
    for index in start:(length(transaction) - (K - depth))
        bucket = mod(transaction[index], tree.bucket_count)
        child_id = get(node.children, bucket, 0)
        if child_id != 0
            _query_hash_node!(workspace, tree, child_id, transaction, index + 1, depth + 1)
        end
    end
    return
end

"""Return matching candidate IDs in workspace-owned storage.

Transactions are sorted, unique vectors of nonnegative Int item IDs, as produced
by the miner's normalization. Create the workspace for the tree being queried.
"""
function matching_ids!(workspace::HashQueryWorkspace, tree::HashTree{K},
                       transaction::Vector{Int}) where {K}
    for index in workspace.touched_items
        workspace.membership[index] = false
    end
    for node_id in workspace.touched_leaves
        workspace.visited_leaves[node_id] = false
    end
    empty!(workspace.touched_items)
    empty!(workspace.touched_leaves)
    empty!(workspace.result_ids)
    for item in transaction
        index = get(tree.item_positions, item, 0)
        if index != 0 && !workspace.membership[index]
            workspace.membership[index] = true
            push!(workspace.touched_items, index)
        end
    end
    if length(transaction) >= K
        _query_hash_node!(workspace, tree, 1, transaction, 1, 1)
    end
    return workspace.result_ids
end

"""Return an independently owned vector of matching candidate IDs."""
function matching_ids(tree::HashTree, transaction::Vector{Int})
    return matching_ids!(HashQueryWorkspace(tree), tree, transaction)
end
''',
    "src/io.jl": r'''"""Read SPMF integer transactions; skip metadata and blank lines."""
function read_spmf(source::IO)
    database = Vector{Int}[]
    for (line_number, line) in enumerate(eachline(source))
            if line_number == 1 && startswith(line, '\ufeff')
                line = chop(line; head=1, tail=0)
            end
            text = strip(line)
            (isempty(text) || startswith(text, ('#', '%', '@'))) && continue
            items = Int[]
            for token in split(text)
                item = tryparse(Int, token)
                (isnothing(item) || item < 0) &&
                    throw(ArgumentError("line $line_number: expected nonnegative integer items"))
                push!(items, item)
            end
            push!(database, unique!(sort!(items)))
    end
    return database
end

read_spmf(path::AbstractString)=open(read_spmf,path,"r")

"""Write all itemsets in deterministic size/lexicographic order."""
function write_spmf(itemsets::AbstractDict, path::AbstractString)
    ordered = sort!(collect(keys(itemsets)); by=items -> (length(items), items))
    open(path, "w") do target
        for itemset in ordered
            println(target, join(itemset, ' '), " #SUP: ", itemsets[itemset])
        end
    end
    return nothing
end
''',
    "src/reports.jl": r'''"""SHA-256 of the implementation files, independent of output or test data."""
function implementation_fingerprint()
    root=dirname(@__DIR__)
    files=[joinpath(root,"Project.toml")]
    append!(files,sort!(filter(path -> endswith(path,".jl"), readdir(@__DIR__;join=true))))
    stream=IOBuffer()
    for path in files
        write(stream,replace(relpath(path,root),'\\'=>'/'),'\0',read(path),'\0')
    end
    return bytes2hex(sha256(take!(stream)))
end

const PHASE_NAMES = Dict(
    :normalize=>"Chuẩn hóa giao dịch", :threshold=>"Tính ngưỡng support",
    :singleton_count=>"Đếm item đơn", :generate_candidates=>"Sinh ứng viên (join + prune)",
    :build_counter=>"Dựng bộ đếm / hash-tree", :count_support=>"Quét giao dịch và đếm support",
    :filter=>"Lọc tập phổ biến", :level_total=>"Tổng thời gian mức k",
    :read_input=>"Đọc và parse dữ liệu", :write_output=>"Ghi kết quả SPMF",
    :warmup_total=>"Các lượt warmup", :mining_total=>"Toàn bộ mine()",
    :cli_total=>"Xử lý CLI đến trước ghi bundle")

elapsed_ms(ns::UInt64)=Float64(ns)/1_000_000
csv_cell(value)="\"" * replace(string(value),'"'=>"\"\"") * "\""
report_path_key(path)=Sys.iswindows() ? lowercase(normpath(abspath(path))) : normpath(abspath(path))

function report_timings(result, read_ns, write_ns, warmup_ns, cli_ns)
    rows=PhaseTiming[PhaseTiming(:read_input,0,read_ns,0,0)]
    push!(rows,PhaseTiming(:warmup_total,0,warmup_ns,0,0))
    append!(rows,result.timings)
    push!(rows,PhaseTiming(:mining_total,0,result.mining_ns,0,length(result.itemsets)))
    push!(rows,PhaseTiming(:write_output,0,write_ns,0,length(result.itemsets)))
    push!(rows,PhaseTiming(:cli_total,0,cli_ns,0,length(result.itemsets)))
    return rows
end

"""Persist a successful run as human-readable and machine-readable artifacts.

The caller owns a freshly created report_dir. Existing report artifacts are
never overwritten. itemsets.txt / trace.txt may already be the explicit CLI
destinations within that directory. Report serialization and input/source
fingerprinting are outside cli_ns and mining_ns; process wall time includes them.
"""
function write_run_report(report_dir::AbstractString,result::MiningResult;
                          metadata::AbstractDict,output_path::AbstractString,
                          explanation_path=nothing,read_ns::UInt64=UInt64(0),
                          write_ns::UInt64=UInt64(0),warmup_ns::UInt64=UInt64(0),
                          cli_ns::UInt64=UInt64(0))
    isdir(report_dir) || throw(ArgumentError("report directory must already exist"))
    report=joinpath(report_dir,"report.txt")
    timings=joinpath(report_dir,"timings.csv")
    metapath=joinpath(report_dir,"metadata.toml")
    snapshot=joinpath(report_dir,"itemsets.txt")
    tracepath=joinpath(report_dir,"trace.txt")
    for path in (report,timings,metapath)
        ispath(path) && throw(ArgumentError("report artifact already exists: $path"))
    end
    same_output=report_path_key(snapshot)==report_path_key(output_path)
    ispath(snapshot) && !same_output && throw(ArgumentError("itemset snapshot already exists"))
    same_trace=!isnothing(explanation_path) && report_path_key(tracepath)==report_path_key(explanation_path)
    !isnothing(explanation_path) && ispath(tracepath) && !same_trace &&
        throw(ArgumentError("trace snapshot already exists"))

    meta=Dict{String,Any}(string(k)=>v for (k,v) in metadata)
    input=abspath(meta["input_path"])
    meta["schema_version"]=1
    meta["input_path"]=input
    input_final_hash=isfile(input) ? open(io -> bytes2hex(sha256(io)),input) : ""
    meta["input_final_sha256"]=input_final_hash
    meta["input_sha256"]=get(meta,"input_sha256",input_final_hash)
    meta["input_bytes"]=get(meta,"input_bytes",isfile(input) ? filesize(input) : -1)
    meta["input_hash_scope"]=get(meta,"input_hash_scope","file_at_report_time")
    meta["input_changed_after_read"]=meta["input_sha256"]!=input_final_hash
    source_final_hash=implementation_fingerprint()
    meta["implementation_final_sha256"]=source_final_hash
    meta["implementation_sha256"]=get(meta,"implementation_sha256",source_final_hash)
    meta["implementation_changed_during_run"]=meta["implementation_sha256"]!=source_final_hash
    meta["report_written_utc"]=Dates.format(now(UTC),dateformat"yyyy-mm-ddTHH:MM:SS.sss") * "Z"
    meta["julia_version"]=string(VERSION)
    meta["os"]=string(Sys.KERNEL)
    meta["architecture"]=string(Sys.ARCH)
    meta["cpu"]=Sys.CPU_NAME
    meta["julia_threads"]=Threads.nthreads()
    meta["cpu_threads"]=Sys.CPU_THREADS
    meta["word_size"]=Sys.WORD_SIZE
    meta["transactions"]=result.transaction_count
    meta["min_support_count"]=result.min_support_count
    meta["itemset_count"]=length(result.itemsets)
    meta["mining_ns"]=Int(result.mining_ns)
    meta["read_ns"]=Int(read_ns)
    meta["write_ns"]=Int(write_ns)
    meta["warmup_ns"]=Int(warmup_ns)
    meta["cli_ns"]=Int(cli_ns)
    meta["timing_policy"]=meta["warmup_runs"]==0 ? "cold_call_may_include_jit" :
                          "warmup_requested_jit_may_remain"
    meta["timing_unit"]="nanoseconds; CSV also includes elapsed_ms"
    meta["timing_scope"]="mining excludes input/output; cli excludes startup and report serialization; nested level/phase rows must not be summed"
    meta["output_path"]=abspath(output_path)
    meta["itemsets_path"]=abspath(snapshot)
    meta["report_path"]=abspath(report)
    meta["timings_path"]=abspath(timings)
    meta["trace_path"]=isnothing(explanation_path) ? "" : abspath(tracepath)
    rows=report_timings(result,read_ns,write_ns,warmup_ns,cli_ns)

    !same_output && write_spmf(result.itemsets,snapshot)
    !isnothing(explanation_path) && !same_trace && cp(explanation_path,tracepath)
    open(timings,"w") do stream
        println(stream,"run_id,k,phase,elapsed_ns,elapsed_ms,candidates,frequent")
        for row in rows
            println(stream,join(csv_cell.(
                (meta["run_id"],row.size,row.phase,row.elapsed_ns,elapsed_ms(row.elapsed_ns),
                 row.candidates,row.frequent)),','))
        end
    end
    open(metapath,"w") do stream
        TOML.print(stream,meta;sorted=true)
    end
    open(report,"w") do stream
        println(stream,"BÁO CÁO MỘT LẦN CHẠY APRIORI")
        for key in ("run_id","started_utc","input_path","input_sha256","input_hash_scope",
                    "input_changed_after_read","implementation_sha256","implementation_changed_during_run",
                    "transactions","minsup_argument","min_support_count","counter",
                    "itemset_count","julia_version","os","architecture","cpu","julia_threads",
                    "warmup_runs","explain_enabled","explain_limit","timing_policy")
            println(stream,key," = ",meta[key])
        end
        println(stream,"\nPHẠM VI ĐO THỜI GIAN")
        println(stream,"Đơn vị: ms để đọc; CSV giữ elapsed_ns gốc. time_ns() là đồng hồ monotonic.")
        println(stream,"read_input gồm đọc snapshot, SHA256 và parse cùng nội dung; mining không gồm bước này.")
        println(stream,"Mining đo mine(); không gồm ghi output hoặc serialize báo cáo.")
        println(stream,"CLI đo xử lý đến trước ghi bundle; không gồm khởi động Julia hoặc ghi bundle.")
        println(stream,"level_total và các phase con lồng nhau: không cộng mọi dòng thành tổng.")
        println(stream,"Thời gian có thể chứa JIT/GC/dispatch. Warmup không bảo đảm bỏ hết JIT.")
        println(stream,"Bật explain có chi phí format/IO trong các phase; chỉ so cùng chế độ.")
        println(stream,"Scan có build_counter = 0; generate_candidates gồm join và prune.")
        println(stream,"Mức cuối C_k rỗng vẫn có timing, nhưng không thêm LevelStats đã đếm giả.")
        println(stream,"Đối chiếu hash cuối và serialize báo cáo nằm ngoài mining/CLI; hash snapshot nằm trong read_input.")
        if meta["input_changed_after_read"] || meta["implementation_changed_during_run"]
            println(stream,"CẢNH BÁO: file input hoặc source đã đổi sau snapshot; metadata giữ hash ban đầu và hash cuối riêng.")
        end
        println(stream,"\nk | phase | mô tả | elapsed_ms | candidates | frequent")
        for row in rows
            println(stream,row.size," | ",row.phase," | ",get(PHASE_NAMES,row.phase,string(row.phase)),
                    " | ",round(elapsed_ms(row.elapsed_ns);digits=6)," | ",row.candidates," | ",row.frequent)
        end
        println(stream,"\nTOÀN BỘ FREQUENT ITEMSETS")
        println(stream,"itemset | support_count | support_relative")
        for itemset in sort!(collect(keys(result.itemsets));by=items -> (length(items),items))
            count=result.itemsets[itemset]
            ratio=iszero(result.transaction_count) ? 0.0 : count/result.transaction_count
            println(stream,item_text(itemset)," | ",count," | ",round(ratio;digits=10))
        end
        isempty(result.itemsets) && println(stream,"(Không có itemset đạt ngưỡng.)")
        println(stream,"\nFile chuẩn SPMF: ",abspath(snapshot))
        println(stream,"Dữ liệu thời gian: ",abspath(timings))
        println(stream,"Metadata: ",abspath(metapath))
        !isnothing(explanation_path) && println(stream,"Luồng chi tiết: ",abspath(tracepath))
        println(stream,"Đây là lần chạy thực tế; chưa phải benchmark lặp lại hoặc đối chiếu SPMF/Java.")
    end
    return report
end
''',
    "src/run_directories.jl": r'''"""Claim a fresh stem-N directory, starting after the largest existing suffix."""
function numbered_run_directory(parent, stem; validate=identity)
    prefix = stem * "-"
    Sys.iswindows() && (prefix = lowercase(prefix))
    last_number = big(0)
    if isdir(parent)
        for entry in readdir(parent; sort=false)
            name = Sys.iswindows() ? lowercase(entry) : entry
            startswith(name, prefix) || continue
            suffix = chopprefix(name, prefix)
            occursin(r"^[0-9]+\z", suffix) || continue
            last_number = max(last_number, parse(BigInt, suffix))
        end
    end
    number = last_number + 1
    while true
        directory = joinpath(parent, "$stem-$number")
        validate(directory)
        mkpath(parent)
        try
            return mkdir(directory)
        catch exc
            # Another process may have claimed this number since readdir.
            exc isa Base.IOError && exc.code == Base.UV_EEXIST || rethrow()
            number += 1
        end
    end
end
''',
    "src/apriori_external.jl": r'''# Disk-backed Apriori path used by the benchmark runner. Itemsets in each level
# are stored as fixed-width Int64 records in lexicographic order.

struct DiskLevel
    path::String
    width::Int
    count::Int
end

struct ExternalMiningSummary
    transaction_count::Int
    min_count::Int
    itemsets::Int
    support_sum::UInt128
    candidate_total::Int
    level_count::Int
    fingerprint_sum::UInt64
    fingerprint_xor::UInt64
end

@inline function external_mix64(value::UInt64)
    value ⊻= value >> 30
    value *= 0xbf58476d1ce4e5b9
    value ⊻= value >> 27
    value *= 0x94d049bb133111eb
    value ⊻= value >> 31
    return value
end

@inline function external_itemset_hash(itemset::NTuple{K,Int}, support::Int) where K
    h = external_mix64(UInt64(K) ⊻ (UInt64(support) << 1))
    for item in itemset
        h = external_mix64(h ⊻ UInt64(item))
    end
    return external_mix64(h ⊻ UInt64(support))
end

@inline function write_itemset(io::IO, itemset::NTuple{K,Int}) where K
    for item in itemset
        write(io, Int64(item))
    end
end

@inline function read_itemset_at(io::IO, level::DiskLevel, index::Int)
    1 <= index <= level.count || throw(BoundsError(1:level.count, index))
    seek(io, (index - 1) * level.width * sizeof(Int64))
    return ntuple(_ -> Int(read(io, Int64)), level.width)
end

@inline function tuple_less(left::Tuple, right::Tuple)
    for i in eachindex(left)
        left[i] == right[i] || return left[i] < right[i]
    end
    return false
end

function disk_level_contains(io::IO, level::DiskLevel, target::NTuple{K,Int}) where K
    low, high = 1, level.count
    while low <= high
        middle = (low + high) >>> 1
        value = read_itemset_at(io, level, middle)
        if value == target
            return true
        elseif tuple_less(value, target)
            low = middle + 1
        else
            high = middle - 1
        end
    end
    return false
end

# Bounded-memory Bloom filter: most absent subsets are rejected without random
# disk seeks. False positives are resolved by the exact binary search above.
struct DiskMembershipBloom
    bits::Vector{UInt64}
    bit_count::UInt64
end

@inline function bloom_hash_pair(itemset::Tuple)
    h = UInt64(0xcbf29ce484222325)
    for item in itemset
        h = (h ⊻ UInt64(item)) * UInt64(0x100000001b3)
    end
    first = external_mix64(h)
    second = external_mix64(first ⊻ UInt64(0x9e3779b97f4a7c15)) | UInt64(1)
    return first, second
end

function build_disk_membership_bloom(level::DiskLevel; max_bytes::Int=32 * 1024^2)
    max_bytes >= 8 || throw(ArgumentError("Bloom filter memory limit must be >= 8 bytes"))
    requested_bits = max(UInt64(64), UInt64(level.count) * UInt64(10))
    maximum_bits = UInt64(max_bytes) * UInt64(8)
    bit_count = min(requested_bits, maximum_bits)
    bit_count = max(UInt64(64), bit_count - (bit_count % UInt64(64)))
    filter = DiskMembershipBloom(zeros(UInt64, Int(bit_count ÷ UInt64(64))), bit_count)
    open(level.path, "r") do source
        for _ in 1:level.count
            itemset = ntuple(_ -> Int(read(source, Int64)), level.width)
            h1, h2 = bloom_hash_pair(itemset)
            index = rem(h1, bit_count)
            step = rem(h2, bit_count)
            for _ in 1:7
                word = Int((index >>> 6) + 1)
                bit = UInt64(1) << (index & UInt64(63))
                filter.bits[word] |= bit
                index += step
                index >= bit_count && (index -= bit_count)
            end
        end
    end
    return filter
end

@inline function bloom_might_contain(filter::DiskMembershipBloom, itemset::Tuple)
    h1, h2 = bloom_hash_pair(itemset)
    index = rem(h1, filter.bit_count)
    step = rem(h2, filter.bit_count)
    for _ in 1:7
        word = Int((index >>> 6) + 1)
        bit = UInt64(1) << (index & UInt64(63))
        (filter.bits[word] & bit) == 0 && return false
        index += step
        index >= filter.bit_count && (index -= filter.bit_count)
    end
    return true
end

# Exact compact membership index for item IDs that fit in a 64- or 128-bit
# packed key. This avoids one disk binary search for every positive Bloom query.
struct PackedDiskMembership{T<:Unsigned}
    keys::Set{T}
    bits_per_item::Int
end

@inline function packed_itemset_key(itemset::Tuple, bits::Int, ::Type{T}) where {T<:Unsigned}
    key = zero(T)
    for item in itemset
        key = (key << bits) | T(item)
    end
    return key
end

@inline function disk_membership_contains(index::PackedDiskMembership{T},
                                          itemset::Tuple) where {T<:Unsigned}
    return packed_itemset_key(itemset, index.bits_per_item, T) in index.keys
end

function build_packed_disk_membership(previous::DiskLevel;
                                      max_bytes::Int=128 * 1024^2)
    previous.count == 0 && return nothing
    max_bytes >= 0 || throw(ArgumentError("packed membership memory limit must be nonnegative"))

    maximum_item = UInt64(0)
    open(previous.path, "r") do source
        for _ in 1:previous.count, _ in 1:previous.width
            value = read(source, Int64)
            value >= 0 || throw(ArgumentError("disk level contains a negative item"))
            maximum_item = max(maximum_item, UInt64(value))
        end
    end
    bits = max(1, 64 - leading_zeros(maximum_item))
    total_bits = previous.width * bits
    key_type, estimated_bytes_per_key = if total_bits <= 64
        (UInt64, 32)
    elseif total_bits <= 128
        (UInt128, 40)
    else
        return nothing
    end
    Int128(previous.count) * estimated_bytes_per_key <= max_bytes || return nothing

    keys = Set{key_type}()
    sizehint!(keys, previous.count)
    open(previous.path, "r") do source
        for _ in 1:previous.count
            itemset = ntuple(_ -> Int(read(source, Int64)), previous.width)
            push!(keys, packed_itemset_key(itemset, bits, key_type))
        end
    end
    return PackedDiskMembership(keys, bits)
end

"""Generate sorted candidates from a disk-backed sorted L_k, with bounded RAM."""
function foreach_disk_candidate_batch(callback::F, previous::DiskLevel,
                                      batch_size::Int; progress::Bool=false,
                                      level::Int=previous.width + 1) where F
    batch_size >= 1 || throw(ArgumentError("candidate_batch_size must be >= 1"))
    K = previous.width
    K >= 1 || throw(ArgumentError("invalid disk level width"))
    batch = NTuple{K+1,Int}[]
    total = 0
    generation_started = time_ns()

    # Small levels use a tuple Set. Larger levels try an exact packed-key Set
    # (capped at 128 MiB); only levels that cannot be packed/fitted use Bloom +
    # exact on-disk binary search.
    cache_limit = 64 * 1024^2
    estimated_cache_bytes = previous.count * (previous.width * sizeof(Int) + 24)
    known = nothing
    packed_known = nothing
    if estimated_cache_bytes <= cache_limit
        cached = Set{NTuple{K,Int}}()
        sizehint!(cached, previous.count)
        open(previous.path, "r") do source
            for _ in 1:previous.count
                push!(cached, ntuple(_ -> Int(read(source, Int64)), K))
            end
        end
        known = cached
    else
        packed_known = build_packed_disk_membership(previous)
    end
    bloom = isnothing(known) && isnothing(packed_known) ?
            build_disk_membership_bloom(previous) : nothing

    progress_step = max(1, cld(previous.count, 20))
    open(previous.path, "r") do sequential_source
        open(previous.path, "r") do membership_source
            for i in 1:previous.count
                left = read_itemset_at(sequential_source, previous, i)
                seek(sequential_source, i * previous.width * sizeof(Int64))
                for j in (i + 1):previous.count
                    right = ntuple(_ -> Int(read(sequential_source, Int64)), K)
                    all(left[p] == right[p] for p in 1:(K - 1)) || break
                    candidate = (left..., right[K])

                    # The two subsets already known from the join are obtained
                    # by removing positions K and K+1. Check every other K-subset.
                    # In particular, removing position 1 must be checked too;
                    # skipping it admits candidates that Apriori should prune.
                    valid = true
                    for removed in 1:(K - 1)
                        subset = ntuple(p -> candidate[p + (p >= removed)], Val(K))
                        present = if !isnothing(known)
                            subset in known
                        elseif !isnothing(packed_known)
                            disk_membership_contains(packed_known, subset)
                        elseif bloom_might_contain(bloom, subset)
                            disk_level_contains(membership_source, previous, subset)
                        else
                            false
                        end
                        if !present
                            valid = false
                            break
                        end
                    end
                    if valid
                        push!(batch, candidate)
                        total += 1
                        if length(batch) == batch_size
                            callback(batch)
                            empty!(batch)
                        end
                    end
                end
                if progress && (i % progress_step == 0 || i == previous.count)
                    pct = round(100 * i / previous.count; digits=1)
                    println(stderr, "[external] L$level join scan: $i/$(previous.count) ($pct%) | candidates=$total")
                    flush(stderr)
                end
            end
        end
    end
    isempty(batch) || callback(batch)
    return (; total, generation_ns=UInt64(time_ns() - generation_started))
end

function create_level_file(directory::String, width::Int, name::String)
    path = joinpath(directory, name * ".bin")
    return path
end

function build_external_summary(transaction_count, threshold, levels, itemsets,
                                support_sum, fingerprint_sum, fingerprint_xor)
    candidate_total = sum(level.candidates for level in levels; init=0)
    return ExternalMiningSummary(transaction_count, threshold, itemsets, support_sum,
        candidate_total, length(levels), fingerprint_sum, fingerprint_xor)
end

"""Shared disk-output loop. `count_batch` returns supports aligned with candidates."""
function external_mine_levels!(directory::String, transaction_count::Int, threshold::Int,
                               singleton_counts::Dict{Int,Int}, count_batch::F,
                               candidate_batch_size::Int; progress::Bool=false, itemset_io=nothing) where F
    levels = LevelStats[]
    transaction_count == 0 && return build_external_summary(transaction_count, threshold,
        levels, 0, UInt128(0), UInt64(0), UInt64(0))
    singleton_total = length(singleton_counts)
    l1_path = create_level_file(directory, 1, "level-1")
    l1_count = 0
    itemset_count = 0
    support_sum = UInt128(0)
    fingerprint_sum = UInt64(0)
    fingerprint_xor = UInt64(0)

    open(l1_path, "w") do output
        for item in sort!(collect(keys(singleton_counts)))
            support = singleton_counts[item]
            if support >= threshold
                itemset = (item,)
                write_itemset(output, itemset)
                isnothing(itemset_io) || println(itemset_io, item, " #SUP: ", support)
                l1_count += 1
                itemset_count += 1
                support_sum += UInt128(support)
                h = external_itemset_hash(itemset, support)
                fingerprint_sum += h
                fingerprint_xor ⊻= h
            end
        end
    end
    push!(levels, LevelStats(1, singleton_total, l1_count))
    empty!(singleton_counts)
    previous = DiskLevel(l1_path, 1, l1_count)
    itemset_size = 2

    while previous.count > 0
        current_path = create_level_file(directory, itemset_size, "level-$itemset_size")
        current_count = Ref(0)
        progress && println(stderr, "[external] Starting L$itemset_size from $(previous.count) itemsets")
        progress && flush(stderr)
        batch_number = Ref(0)
        counted_candidates = Ref(0)
        generation_phase = open(current_path, "w") do output
            foreach_disk_candidate_batch(previous, candidate_batch_size;
                                         progress, level=itemset_size) do candidates
                batch_number[] += 1
                counted_candidates[] += length(candidates)
                supports = count_batch(candidates, itemset_size)
                if progress && (batch_number[] == 1 || batch_number[] % 10 == 0)
                    println(stderr, "[external] L$itemset_size support-count batch $(batch_number[]) | candidates=$(counted_candidates[]) | frequent=$(current_count[])")
                    flush(stderr)
                end
                length(supports) == length(candidates) || error("counter returned wrong support count")
                for id in eachindex(candidates)
                    support = supports[id]
                    if support >= threshold
                        itemset = candidates[id]
                        write_itemset(output, itemset)
                        isnothing(itemset_io) || println(itemset_io, join(itemset, ' '), " #SUP: ", support)
                        current_count[] += 1
                        itemset_count += 1
                        support_sum += UInt128(support)
                        h = external_itemset_hash(itemset, support)
                        fingerprint_sum += h
                        fingerprint_xor ⊻= h
                    end
                end
            end
        end
        if generation_phase.total == 0
            rm(current_path; force=true)
            break
        end
        push!(levels, LevelStats(itemset_size, generation_phase.total, current_count[]))
        progress && println(stderr, "[external] Finished L$itemset_size | candidates=$(generation_phase.total) | frequent=$(current_count[]) | elapsed=$(round(generation_phase.generation_ns / 1e9; digits=2))s")
        progress && flush(stderr)
        old_level_path = previous.path
        previous = DiskLevel(current_path, itemset_size, current_count[])
        rm(old_level_path; force=true)
        itemset_size += 1
    end
    return build_external_summary(transaction_count, threshold, levels, itemset_count,
                                  support_sum, fingerprint_sum, fingerprint_xor)
end

function with_external_directory(callback::F, workdir) where F
    directory = isnothing(workdir) ? mktempdir(prefix="fim-apriori-") : mktempdir(workdir; prefix="fim-apriori-")
    try
        return callback(directory)
    finally
        rm(directory; recursive=true, force=true)
    end
end

"""Baseline Apriori benchmark path: input transactions stay in RAM; levels/results stay on disk."""
function mine_external(transactions; min_support=nothing, min_count=nothing,
                       counter=:hash_tree, candidate_batch_size::Int=10_000,
                       workdir=nothing, normalized::Bool=false, progress::Bool=false, itemset_io=nothing)
    candidate_batch_size >= 1 || throw(ArgumentError("candidate_batch_size must be >= 1"))
    mode = counter_mode(counter)
    database = if normalized
        transactions isa Vector{Vector{Int}} ||
            throw(ArgumentError("normalized=true requires Vector{Vector{Int}} input"))
        transactions
    else
        normalize_transactions(transactions)
    end
    threshold = support_threshold(length(database), min_support, min_count)
    singleton_counts = Dict{Int,Int}()
    for transaction in database, item in transaction
        singleton_counts[item] = get(singleton_counts, item, 0) + 1
    end
    return with_external_directory(workdir) do directory
        count_batch = (candidates, _) -> count_candidates(candidates, database, mode)
        external_mine_levels!(directory, length(database), threshold, singleton_counts,
                              count_batch, candidate_batch_size; progress, itemset_io)
    end
end

function mine_bitset_external(path::AbstractString; min_support=nothing, min_count=nothing,
                              candidate_batch_size::Int=10_000, workdir=nothing,
                              progress::Bool=false, itemset_io=nothing)
    candidate_batch_size >= 1 || throw(ArgumentError("candidate_batch_size must be >= 1"))
    transaction_count, singleton_counts = file_singleton_counts(path)
    threshold = support_threshold(transaction_count, min_support, min_count)
    return with_external_directory(workdir) do directory
        # Build the L1 disk file first, then construct vertical bitmaps from its IDs.
        l1_items = sort!([item for (item, count) in singleton_counts if count >= threshold])
        columns = Dict(item => column for (column, item) in enumerate(l1_items))
        bitmaps = zeros(UInt64, cld(transaction_count, 64), length(l1_items))
        observed = foreach_spmf_transaction(path) do tid, transaction
            word_index = ((tid - 1) >>> 6) + 1
            bit_index = (tid - 1) & 63
            bit = one(UInt64) << bit_index
            for item in transaction
                column = get(columns, item, 0)
                column == 0 || (bitmaps[word_index, column] |= bit)
            end
        end
        observed == transaction_count || throw(ArgumentError("input file changed between passes"))
        count_batch = (candidates, _) -> begin
            counts = Vector{Int}(undef, length(candidates))
            for id in eachindex(candidates)
                counts[id] = bitset_support(bitmaps, columns, candidates[id])
            end
            counts
        end
        external_mine_levels!(directory, transaction_count, threshold, singleton_counts,
                              count_batch, candidate_batch_size; progress, itemset_io)
    end
end

function mine_streaming_external(path::AbstractString; min_support=nothing, min_count=nothing,
                                 candidate_batch_size::Int=10_000, workdir=nothing,
                                 progress::Bool=false, itemset_io=nothing)
    candidate_batch_size >= 1 || throw(ArgumentError("candidate_batch_size must be >= 1"))
    transaction_count, singleton_counts = file_singleton_counts(path)
    threshold = support_threshold(transaction_count, min_support, min_count)
    return with_external_directory(workdir) do directory
        count_batch = (candidates, _) -> begin
            tree = HashTree(candidates)
            workspace = HashQueryWorkspace(tree)
            counts = zeros(Int, length(candidates))
            observed = foreach_spmf_transaction(path) do _, transaction
                for id in matching_ids!(workspace, tree, transaction)
                    counts[id] += 1
                end
            end
            observed == transaction_count || throw(ArgumentError("input file changed between passes"))
            counts
        end
        external_mine_levels!(directory, transaction_count, threshold, singleton_counts,
                              count_batch, candidate_batch_size; progress, itemset_io)
    end
end
''',
    "src/apriori_optimized.jl": r'''# Hai biến thể Apriori tối ưu được thiết kế dựa trên mine() hiện tại.
# 1) mine_bitset: lưu vertical TID bitmap và đếm support bằng AND + popcount.
# 2) mine_streaming: không giữ toàn bộ database trong RAM; đọc file theo từng lượt.

"""Duyệt file SPMF theo từng transaction, không materialize toàn bộ database.

Callback nhận `(tid, transaction)`. Quy tắc bỏ qua dòng trắng/metadata, xử lý BOM,
parse Int, sort và loại item trùng được giữ giống `read_spmf`.
"""
function foreach_spmf_transaction(callback::F, path::AbstractString) where {F}
    transaction_count = 0
    open(path, "r") do source
        for (line_number, line) in enumerate(eachline(source))
            if line_number == 1 && startswith(line, '\ufeff')
                line = chop(line; head=1, tail=0)
            end
            text = strip(line)
            (isempty(text) || startswith(text, ('#', '%', '@'))) && continue

            items = Int[]
            for token in split(text)
                item = tryparse(Int, token)
                (isnothing(item) || item < 0) &&
                    throw(ArgumentError("line $line_number: expected nonnegative integer items"))
                push!(items, item)
            end
            unique!(sort!(items))
            transaction_count += 1
            callback(transaction_count, items)
        end
    end
    return transaction_count
end

"""First file pass: count N and singleton support without retaining transactions."""
function file_singleton_counts(path::AbstractString)
    singletons = Dict{Int,Int}()
    transaction_count = foreach_spmf_transaction(path) do _, transaction
        for item in transaction
            singletons[item] = get(singletons, item, 0) + 1
        end
    end
    return transaction_count, singletons
end

"""Create L1 and fill the common MiningResult containers."""
function initial_frequent_level!(answer, levels, singletons, threshold)
    previous = NTuple{1,Int}[(item,) for (item, count) in singletons if count >= threshold]
    sort!(previous)
    for itemset in previous
        answer[itemset] = singletons[itemset[1]]
    end
    push!(levels, LevelStats(1, length(singletons), length(previous)))
    return previous
end

"""Count an itemset by intersecting its singleton TID bitmaps.

`bitmaps` has one UInt64 column per frequent singleton and one row per 64 TIDs.
The candidate is mapped to bitmap columns once, outside the word loop.
"""
@inline function bitset_support(bitmaps::Matrix{UInt64}, columns::Dict{Int,Int},
                                candidate::NTuple{K,Int}) where {K}
    candidate_columns = ntuple(index -> columns[candidate[index]], Val(K))
    support = 0
    @inbounds for word_index in axes(bitmaps, 1)
        intersection = bitmaps[word_index, candidate_columns[1]]
        for position in 2:K
            intersection &= bitmaps[word_index, candidate_columns[position]]
            iszero(intersection) && break
        end
        support += count_ones(intersection)
    end
    return support
end

"""Build a UInt64 vertical bitmap matrix for items in L1."""
function build_singleton_bitmaps(path::AbstractString, transaction_count::Int,
                                 previous::Vector{NTuple{1,Int}})
    columns = Dict{Int,Int}()
    for (column, itemset) in enumerate(previous)
        columns[itemset[1]] = column
    end

    word_count = cld(transaction_count, 64)
    bitmaps = zeros(UInt64, word_count, length(previous))
    observed = foreach_spmf_transaction(path) do tid, transaction
        word_index = ((tid - 1) >>> 6) + 1
        bit_index = (tid - 1) & 63
        bit = one(UInt64) << bit_index
        for item in transaction
            column = get(columns, item, 0)
            if column != 0
                bitmaps[word_index, column] |= bit
            end
        end
    end
    observed == transaction_count ||
        throw(ArgumentError("input file changed between Apriori passes (transaction count differs)"))
    return bitmaps, columns
end

"""Apriori using vertical TID bitmaps; optimized for fast support counting.

The input is a path to an SPMF transaction file. Only items frequent at level 1
receive bitmaps. Candidate support is computed by word-wise intersections and
`count_ones`, avoiding a transaction-by-candidate hash-tree traversal.
"""
function mine_bitset(path::AbstractString; min_support=nothing, min_count=nothing,
                      candidate_batch_size::Int=10_000)
    candidate_batch_size >= 1 ||
        throw(ArgumentError("candidate_batch_size must be >= 1"))
    mining_started = time_ns()
    timings = PhaseTiming[]

    started = time_ns()
    transaction_count, singletons = file_singleton_counts(path)
    record_phase!(timings, :singleton_count, 1, time_ns() - started, length(singletons), 0)

    started = time_ns()
    threshold = support_threshold(transaction_count, min_support, min_count)
    record_phase!(timings, :threshold, 0, time_ns() - started, 0, 0)

    answer = Dict{Tuple{Vararg{Int}},Int}()
    levels = LevelStats[]
    if iszero(transaction_count)
        return MiningResult(answer, transaction_count, threshold, levels,
                            timings, time_ns() - mining_started)
    end
    previous = initial_frequent_level!(answer, levels, singletons, threshold)
    isempty(previous) && return MiningResult(answer, transaction_count, threshold, levels,
                                               timings, time_ns() - mining_started)

    # L1's support counts are now copied to the result; release the temporary map.
    singletons = nothing  # let the singleton dictionary become collectible
    # With one frequent singleton, C2 is empty; retain the original phase record
    # but skip the unnecessary second file pass that would build a one-column bitmap.
    if length(previous) < 2
        level_started = time_ns()
        started = time_ns()
        apriori_gen(previous)
        record_phase!(timings, :generate_candidates, 2, time_ns() - started, 0, 0)
        record_phase!(timings, :level_total, 2, time_ns() - level_started, 0, 0)
        return MiningResult(answer, transaction_count, threshold, levels,
                            timings, time_ns() - mining_started)
    end

    started = time_ns()
    bitmaps, columns = build_singleton_bitmaps(path, transaction_count, previous)
    record_phase!(timings, :build_counter, 1, time_ns() - started,
                  length(previous), 0)

    itemset_size = 2
    while !isempty(previous)
        level_started = time_ns()
        candidate_type = typeof((first(previous)..., 0))
        current = candidate_type[]
        generation_phase_index = Ref(0)
        candidate_stats = foreach_candidate_batch(previous, candidate_batch_size, NoTrace()) do candidates
            if iszero(generation_phase_index[])
                push!(timings, PhaseTiming(:generate_candidates, itemset_size, UInt64(0), 0, 0))
                generation_phase_index[] = length(timings)
            end
            count_started = time_ns()
            counts = Vector{Int}(undef, length(candidates))
            for candidate_id in eachindex(candidates)
                counts[candidate_id] = bitset_support(bitmaps, columns, candidates[candidate_id])
            end
            record_phase!(timings, :count_support, itemset_size,
                          time_ns() - count_started, length(candidates), 0)
            filter_started = time_ns()
            frequent_in_batch = 0
            for candidate_id in eachindex(candidates)
                support = counts[candidate_id]
                if support >= threshold
                    candidate = candidates[candidate_id]
                    push!(current, candidate)
                    answer[candidate] = support
                    frequent_in_batch += 1
                end
            end
            record_phase!(timings, :filter, itemset_size, time_ns() - filter_started,
                          length(candidates), frequent_in_batch)
        end
        candidate_total = candidate_stats.total
        if iszero(generation_phase_index[])
            record_phase!(timings, :generate_candidates, itemset_size,
                          candidate_stats.generation_ns, candidate_total, 0)
        else
            timings[generation_phase_index[]] =
                PhaseTiming(:generate_candidates, itemset_size, candidate_stats.generation_ns,
                            candidate_total, 0)
        end
        if iszero(candidate_total)
            record_phase!(timings, :level_total, itemset_size,
                          time_ns() - level_started, 0, 0)
            break
        end
        push!(levels, LevelStats(itemset_size, candidate_total, length(current)))
        record_phase!(timings, :level_total, itemset_size, time_ns() - level_started,
                      candidate_total, length(current))
        previous = current
        itemset_size += 1
    end

    return MiningResult(answer, transaction_count, threshold, levels, timings,
                        time_ns() - mining_started)
end

"""Apriori with a disk-backed transaction source and bounded transaction memory.

The file is scanned once to count singletons and once per candidate batch for
support counting. At most one normalized transaction is resident while support
is counted. This exchanges additional sequential disk reads for a much smaller
database-memory footprint. Candidate processing is bounded by
`candidate_batch_size`: each batch gets its own hash-tree and counters, then is
discarded. The previous frequent level and final output still reside in RAM.
"""
function mine_streaming(path::AbstractString; min_support=nothing, min_count=nothing,
                         candidate_batch_size::Int=10_000)
    candidate_batch_size >= 1 ||
        throw(ArgumentError("candidate_batch_size must be >= 1"))
    mining_started = time_ns()
    timings = PhaseTiming[]

    started = time_ns()
    transaction_count, singletons = file_singleton_counts(path)
    record_phase!(timings, :singleton_count, 1, time_ns() - started, length(singletons), 0)

    started = time_ns()
    threshold = support_threshold(transaction_count, min_support, min_count)
    record_phase!(timings, :threshold, 0, time_ns() - started, 0, 0)

    answer = Dict{Tuple{Vararg{Int}},Int}()
    levels = LevelStats[]
    if iszero(transaction_count)
        return MiningResult(answer, transaction_count, threshold, levels,
                            timings, time_ns() - mining_started)
    end
    previous = initial_frequent_level!(answer, levels, singletons, threshold)
    isempty(previous) && return MiningResult(answer, transaction_count, threshold, levels,
                                               timings, time_ns() - mining_started)
    singletons = nothing  # let the singleton dictionary become collectible

    itemset_size = 2
    while !isempty(previous)
        level_started = time_ns()
        candidate_type = typeof((first(previous)..., 0))
        current = candidate_type[]
        generation_phase_index = Ref(0)
        candidate_stats = foreach_candidate_batch(previous, candidate_batch_size, NoTrace()) do candidates
            if iszero(generation_phase_index[])
                push!(timings, PhaseTiming(:generate_candidates, itemset_size, UInt64(0), 0, 0))
                generation_phase_index[] = length(timings)
            end
            build_started = time_ns()
            tree = HashTree(candidates)
            workspace = HashQueryWorkspace(tree)
            record_phase!(timings, :build_counter, itemset_size,
                          time_ns() - build_started, length(candidates), 0)

            count_started = time_ns()
            counts = zeros(Int, length(candidates))
            observed = foreach_spmf_transaction(path) do _, transaction
                for candidate_id in matching_ids!(workspace, tree, transaction)
                    counts[candidate_id] += 1
                end
            end
            observed == transaction_count ||
                throw(ArgumentError("input file changed between Apriori passes (transaction count differs)"))
            record_phase!(timings, :count_support, itemset_size,
                          time_ns() - count_started, length(candidates), 0)

            filter_started = time_ns()
            frequent_in_batch = 0
            for candidate_id in eachindex(candidates)
                support = counts[candidate_id]
                if support >= threshold
                    candidate = candidates[candidate_id]
                    push!(current, candidate)
                    answer[candidate] = support
                    frequent_in_batch += 1
                end
            end
            record_phase!(timings, :filter, itemset_size, time_ns() - filter_started,
                          length(candidates), frequent_in_batch)
        end
        candidate_total = candidate_stats.total
        if iszero(generation_phase_index[])
            record_phase!(timings, :generate_candidates, itemset_size,
                          candidate_stats.generation_ns, candidate_total, 0)
        else
            timings[generation_phase_index[]] =
                PhaseTiming(:generate_candidates, itemset_size, candidate_stats.generation_ns,
                            candidate_total, 0)
        end
        if iszero(candidate_total)
            record_phase!(timings, :level_total, itemset_size,
                          time_ns() - level_started, 0, 0)
            break
        end
        push!(levels, LevelStats(itemset_size, candidate_total, length(current)))
        record_phase!(timings, :level_total, itemset_size, time_ns() - level_started,
                      candidate_total, length(current))
        previous = current
        itemset_size += 1
    end

    return MiningResult(answer, transaction_count, threshold, levels, timings,
                        time_ns() - mining_started)
end
''',
    "src/AssociationRules.jl": r'''"""
    AssociationRules

Sinh luật kết hợp `X => Y` từ các frequent itemset đã khai phá bằng Apriori.

Đầu vào là file itemsets theo định dạng SPMF (`1 3 5 #SUP: 42`) và tổng số
transaction `N`. Với mỗi itemset phổ biến `Z` và mỗi cách tách `Z = X ∪ Y`
(X, Y khác rỗng, rời nhau):

    support(X ⇒ Y)    = sup(Z) / N
    confidence(X ⇒ Y) = sup(Z) / sup(X)
    lift(X ⇒ Y)       = confidence / (sup(Y) / N) = sup(Z)·N / (sup(X)·sup(Y))

Do tính chất Apriori, mọi tập con của `Z` đều phổ biến nên `sup(X)` và `sup(Y)`
luôn có sẵn trong file itemsets; không cần quét lại dữ liệu.

Module chỉ dùng thư viện chuẩn của Julia và không có trạng thái toàn cục.
Kết quả được sắp theo thứ tự toàn phần (lift ↓, confidence ↓, count ↓, rồi
antecedent/consequent tăng dần) nên chạy lại luôn cho cùng một đầu ra.
"""
module AssociationRules

export Rule, read_itemsets, count_transactions, generate_rules, write_rules_csv

"Số item tối đa của một itemset được tách luật (2^k phép tách)."
const MAX_ITEMSET_SIZE = 24
const SUP_MARK = "#SUP:"
const EPS = 1e-12

"Một luật kết hợp `antecedent ⇒ consequent` cùng các chỉ số đo."
struct Rule
    antecedent::Vector{Int}
    consequent::Vector{Int}
    count::Int          # sup(X ∪ Y), số transaction chứa cả hai vế
    support::Float64
    confidence::Float64
    lift::Float64
end

"""
    read_itemsets(path) -> Dict{Vector{Int},Int}

Đọc file itemsets SPMF. Khóa là vector item đã sắp tăng, giá trị là support count.
Bỏ qua dòng trống và dòng metadata bắt đầu bằng `@`, `%` hoặc `#`.
"""
function read_itemsets(path::AbstractString)
    counts = Dict{Vector{Int},Int}()
    open(path, "r") do source
        for (ln, line) in enumerate(eachline(source))
            s = strip(line)
            (isempty(s) || s[1] in ('@', '%', '#')) && continue
            pos = findfirst(SUP_MARK, s)
            pos === nothing && throw(ArgumentError("$path:$ln: thiếu '$SUP_MARK'"))
            left = strip(s[1:prevind(s, first(pos))])
            right = strip(s[(last(pos) + 1):end])
            items = try
                sort!(parse.(Int, split(left)))
            catch
                throw(ArgumentError("$path:$ln: item không phải số nguyên"))
            end
            isempty(items) && continue
            counts[items] = try
                parse(Int, right)
            catch
                throw(ArgumentError("$path:$ln: support count không hợp lệ"))
            end
        end
    end
    return counts
end

"""
    count_transactions(path) -> Int

Đếm số transaction hợp lệ của file dữ liệu SPMF (bỏ dòng trống và metadata).
"""
function count_transactions(path::AbstractString)
    n = 0
    for line in eachline(path)
        s = strip(line)
        (isempty(s) || s[1] in ('#', '%', '@')) && continue
        n += 1
    end
    return n
end

function _rule_isless(a::Rule, b::Rule)
    a.lift != b.lift && return a.lift > b.lift
    a.confidence != b.confidence && return a.confidence > b.confidence
    a.count != b.count && return a.count > b.count
    a.antecedent != b.antecedent && return a.antecedent < b.antecedent
    return a.consequent < b.consequent
end

function _rules_from_itemset!(rules::Vector{Rule}, counts::Dict{Vector{Int},Int},
                              items::Vector{Int}, supxy::Int, ntrans::Int,
                              minconf::Float64, minlift::Float64)
    k = length(items)
    full = (1 << k) - 1
    @inbounds for mask in 1:(full - 1)      # mọi tập con khác rỗng và khác toàn bộ
        ant = Int[]
        con = Int[]
        for j in 1:k
            if (mask >> (j - 1)) & 1 == 1
                push!(ant, items[j])
            else
                push!(con, items[j])
            end
        end
        supx = get(counts, ant, 0)
        supy = get(counts, con, 0)
        if supx == 0 || supy == 0
            throw(ArgumentError("thiếu support của tập con $(supx == 0 ? ant : con) " *
                                "trong file itemsets (file không đầy đủ?)"))
        end
        conf = supxy / supx
        conf + EPS < minconf && continue
        lift = Float64(supxy * ntrans) / Float64(supx * supy)
        lift + EPS < minlift && continue
        push!(rules, Rule(ant, con, supxy, supxy / ntrans, conf, lift))
    end
    return rules
end

"""
    generate_rules(counts, ntrans; minconf=0.5, minlift=0.0) -> Vector{Rule}

Sinh mọi luật có confidence ≥ `minconf` và lift ≥ `minlift`, đã sắp xếp
(lift giảm dần, hòa thì confidence, count, rồi thứ tự từ điển).
"""
function generate_rules(counts::Dict{Vector{Int},Int}, ntrans::Integer;
                        minconf::Real=0.5, minlift::Real=0.0)
    ntrans > 0 || throw(ArgumentError("ntrans phải > 0"))
    0 <= minconf <= 1 || throw(ArgumentError("minconf phải nằm trong [0, 1]"))
    minlift >= 0 || throw(ArgumentError("minlift phải >= 0"))
    rules = Rule[]
    for (items, supxy) in counts
        length(items) < 2 && continue
        if length(items) > MAX_ITEMSET_SIZE
            throw(ArgumentError("itemset có $(length(items)) item, vượt giới hạn " *
                                "$MAX_ITEMSET_SIZE; hãy tăng minsup"))
        end
        _rules_from_itemset!(rules, counts, items, supxy, Int(ntrans),
                             Float64(minconf), Float64(minlift))
    end
    sort!(rules; lt=_rule_isless)
    return rules
end

"Ghi luật ra CSV: antecedent, consequent (item cách nhau bằng khoảng trắng), count, support, confidence, lift."
function write_rules_csv(path::AbstractString, rules::Vector{Rule})
    dir = dirname(path)
    isempty(dir) || mkpath(dir)
    open(path, "w") do io
        println(io, "antecedent,consequent,count,support,confidence,lift")
        for r in rules
            println(io, join(r.antecedent, ' '), ',', join(r.consequent, ' '), ',',
                    r.count, ',', r.support, ',', r.confidence, ',', r.lift)
        end
    end
    return path
end

end # module
''',
}
for relative, content in source_files.items():
    target = project / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("w", encoding="utf-8", newline="\n") as stream:
        stream.write(content)
print(f"Đã chuẩn bị {len(source_files)} tệp của dự án.")

In [ ]:
# @title Mã nguồn Julia của Ứng dụng 2 (mở ô để xem cách mã hóa và đếm)
splice_app = project / "apps" / "splice-junction" / "main.jl"
splice_app.parent.mkdir(parents=True, exist_ok=True)
splice_source = r'''module SpliceJunction

import FrequentItemsetMining as FIM
using Printf: @printf

const ROOT = normpath(joinpath(@__DIR__, "..", ".."))
const GROUPS = ("EI", "IE", "N")
const BASES = "ACGT"
const POSITIONS = (-4, -3, -2, -1, 1, 2, 3, 4)
const HELP = """
    julia --project=. apps/splice-junction/main.jl [--minsup 0.20] [--input splice.data]
Apriori xét 4 nucleotide mỗi bên tâm đoạn. Minsup thuộc (0, 1] và áp dụng riêng cho EI, IE, N.
Kết quả mỗi lần chạy được lưu vào thư mục mới bên trong output/splice-junction/.
"""

function encode(sequence)
    (ncodeunits(sequence) == 60 && all(c -> c in "ACGTDRSN", sequence)) ||
        throw(ArgumentError("cần đúng 60 nucleotide theo bảng ký hiệu của bộ UCI"))
    items = Int[]
    # The boundary is between columns 30 and 31. No position is numbered zero.
    for (slot, base) in enumerate(sequence[27:34])
        index = findfirst(==(base), BASES)
        isnothing(index) || push!(items, 4 * (slot - 1) + index)
    end
    return items
end

function item_label(id)
    slot, base = divrem(id - 1, 4)
    pos = POSITIONS[slot + 1]
    return string(pos > 0 ? "+" : "", pos, ":", BASES[base + 1])
end

function read_groups(path)
    groups = Dict(g => Vector{Int}[] for g in GROUPS)
    open(path) do io
        for (line_num, line) in enumerate(eachline(io))
            isempty(strip(line)) && continue
            fields = strip.(split(line, ','))
            (length(fields) == 3 && fields[1] in GROUPS && !isempty(fields[2])) ||
                throw(ArgumentError("dòng $line_num cần có nhãn EI/IE/N, tên và chuỗi DNA"))
            push!(groups[fields[1]], encode(uppercase(fields[3])))
        end
    end
    all(g -> !isempty(groups[g]), GROUPS) || throw(ArgumentError("dữ liệu cần có đủ ba nhóm EI, IE, N"))
    return groups
end

"""The 32 position/base items fit in one mask. Used only to recount across groups."""
function item_mask(items)
    mask = zero(UInt32)
    for id in items
        mask |= UInt32(1) << (id - 1)
    end
    return mask
end

function compare_groups(groups, minsup)
    results = [FIM.mine(groups[g]; min_support=minsup) for g in GROUPS]
    patterns = union((keys(r.itemsets) for r in results)...)
    masks = [item_mask.(groups[g]) for g in GROUPS]
    ordered = sort!(collect(patterns); by=p -> (length(p), p))
    return map(ordered) do pattern
        wanted = item_mask(pattern)
        counts = ntuple(3) do i
            # Missing from a group's frequent sets means below cutoff, possibly nonzero.
            get(results[i].itemsets, pattern) do
                count(mask -> (mask & wanted) == wanted, masks[i])
            end
        end
        support = ntuple(i -> counts[i] // results[i].transaction_count, 3)
        (; pattern, counts, support)
    end
end

pattern_label(p) = join(item_label.(p), " ")

function write_patterns(path, rows)
    open(path, "w") do io
        println(io, "pattern,size,count_EI,count_IE,count_N,support_EI,support_IE,support_N")
        for row in rows
            print(io, pattern_label(row.pattern), ',', length(row.pattern), ',', join(row.counts, ','))
            for value in row.support
                @printf(io, ",%.8f", value)
            end
            println(io)
        end
    end
end

function write_report(path, groups, minsup, rows)
    cutoff = FIM.support_ratio(minsup)
    open(path, "w") do io
        println(io, "Các mẫu nucleotide quanh điểm nối exon – intron")
        println(io, "Số đoạn: ", join(("$g=$(length(groups[g]))" for g in GROUPS), ", "))
        println(io, "Vị trí: -4..-1, +1..+4. Minsup mỗi nhóm: $minsup. Tổng số mẫu khác nhau: ", length(rows))
        for (i, boundary) in enumerate(("exon → intron", "intron → exon"))
            group = GROUPS[i]
            pairs = filter(r -> length(r.pattern) == 2 && r.support[i] >= cutoff &&
                                r.support[i] > r.support[3], rows)
            println(io, "\n$group ($boundary)")
            if isempty(pairs)
                println(io, "Không có cặp đạt ngưỡng trong $group và phổ biến hơn ở nhóm N.")
                continue
            end
            # The CSV contains all patterns. Choose one pair per boundary for interpretation.
            row = first(sort!(pairs; by=r -> (r.support[3] - r.support[i], -r.support[i], r.pattern)))
            println(io, "Cặp có chênh lệch support lớn nhất so với N: ", pattern_label(row.pattern))
            for (j, g) in enumerate(GROUPS)
                @printf(io, "  %s: %d/%d (%.2f%%)\n", g, row.counts[j], length(groups[g]), 100 * row.support[j])
            end
            println(io, "Trong bộ UCI, hai nucleotide ở đúng hai vị trí này thường đi cùng nhau gần ranh giới $boundary hơn ở tâm các đoạn đối chứng.")
        end
        println(io, "\nCác tỷ lệ mô tả tập dữ liệu đã gán nhãn. Đánh giá khả năng dự đoán điểm nối cần dữ liệu kiểm chứng độc lập.")
    end
end

function main(args=ARGS; output_root=joinpath(ROOT,"output","splice-junction"), out=stdout, err=stderr)
    if args == ["--help"]
        print(out, HELP)
        return 0
    end
    try
        flags = Dict{String,String}()
        for i in 1:2:length(args)
            key = args[i]
            (key in ("--minsup", "--input") && !haskey(flags, key) && i < length(args)) ||
                throw(ArgumentError("tham số không hợp lệ: $key. Xem --help"))
            flags[key] = args[i+1]
        end
        minsup = get(flags, "--minsup", "0.20")
        FIM.support_threshold(1, minsup, nothing)
        path = get(flags, "--input", joinpath(ROOT,"data","splice-junction","splice.data"))
        groups = read_groups(path)
        rows = compare_groups(groups, minsup)
        dir = FIM.numbered_run_directory(output_root, "splice")
        csv, report = joinpath(dir, "patterns.csv"), joinpath(dir, "report.txt")
        try
            write_patterns(csv, rows)
            write_report(report, groups, minsup, rows)
        catch
            rm(csv; force=true)
            rm(report; force=true)
            isempty(readdir(dir)) && rm(dir)
            rethrow()
        end
        println(out, join(("$g=$(length(groups[g]))" for g in GROUPS), ", "), "; mẫu: ", length(rows))
        println(out, "CSV: ", csv)
        println(out, "Báo cáo: ", report)
        return 0
    catch exc
        if exc isa ArgumentError || exc isa SystemError || exc isa Base.IOError
            println(err, "splice-junction: ", sprint(showerror, exc))
            return 2
        end
        rethrow()
    end
end

end # module

if abspath(PROGRAM_FILE) == @__FILE__
    exit(SpliceJunction.main())
end
'''
with splice_app.open("w", encoding="utf-8", newline="\n") as stream:
    stream.write(splice_source)

In [ ]:
splice_input = verified_download(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/splice-junction-gene-sequences/splice.data",
    cache / "splice.data",
    "ebbae10c85d3c285e2a2489a37e42bc2ee8d91d15005e391a9962fa3389a114f",
)
from collections import Counter

with splice_input.open(encoding="ascii", newline="") as stream:
    splice_sizes = Counter(row[0].strip() for row in csv.reader(stream))
print("Số đoạn theo nhãn:", dict(splice_sizes))

## Mã hóa theo vị trí và khai phá

Một đoạn là một giao dịch. Xét 4 nucleotide trước và 4 nucleotide sau tâm, giữa cột 30 và 31. Ví dụ, `-2:A` nghĩa là có A ở vị trí thứ hai bên trái. Mỗi cặp vị trí–base có một ID cố định trong cả ba nhóm. Một đoạn chứa ký hiệu mơ hồ vẫn thuộc mẫu số của nhóm, chỉ vị trí mơ hồ được bỏ qua. Hàm Julia `encode` trong ô mã nguồn thực hiện chuyển đổi từ chuỗi gốc.

Apriori của nhóm chạy độc lập trên EI, IE và N. Một mẫu xuất hiện trong CSV nếu đạt ngưỡng ở ít nhất một nhóm. Với từng mẫu đó, tính support ở cả ba nhóm bằng số đoạn chứa mẫu chia cho tổng số đoạn của nhóm.

In [ ]:
# @title Chạy Apriori cho ba nhóm DNA
splice_minsup = "0.20"  # @param {type:"string"}
splice_run = subprocess.run(
    [julia, "--startup-file=no", "--project=.", str(splice_app), "--input",
     str(splice_input), "--minsup", splice_minsup],
    cwd=project, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
)
print(splice_run.stdout)
splice_run.check_returncode()
splice_csv = Path(next(line.removeprefix("CSV: ") for line in splice_run.stdout.splitlines()
                       if line.startswith("CSV: ")))

## Đối chiếu support giữa ba nhóm

Bảng dưới hiển thị tối đa 5 cặp cho mỗi loại ranh giới. Cặp được chọn khi đạt minsup trong nhóm đó và có support cao hơn ở nhóm N. Xếp các cặp theo mức chênh so với N, tính bằng điểm phần trăm. Tệp CSV chứa toàn bộ mẫu, kể cả mẫu đơn và mẫu dài hơn hai vị trí.

In [ ]:
from fractions import Fraction
from math import ceil
import pandas as pd

splice_patterns = pd.read_csv(splice_csv)
views = []
for group in ("EI", "IE"):
    target = splice_patterns["count_" + group]
    background = splice_patterns["count_N"]
    selected = splice_patterns.loc[
        (splice_patterns["size"] == 2) &
        (target >= ceil(Fraction(splice_minsup) * splice_sizes[group])) &
        (target * splice_sizes["N"] > background * splice_sizes[group])
    ].copy()
    selected["group"] = group
    selected["gap_pp"] = 100 * (selected["count_" + group] / splice_sizes[group] -
                                selected["count_N"] / splice_sizes["N"])
    views.append(selected.sort_values("gap_pp", ascending=False, kind="stable").head(5))

view = pd.concat(views, ignore_index=True)[
    ["group","pattern","count_EI","count_IE","count_N","support_EI","support_IE","support_N","gap_pp"]
]
for column in ("support_EI", "support_IE", "support_N"):
    view[column] = view[column].map("{:.2%}".format)
view.round({"gap_pp": 2})

In [ ]:
print(splice_csv.with_name("report.txt").read_text(encoding="utf-8"))

### Ý nghĩa sinh học ở ngưỡng mặc định

Cặp G ở +1 và T ở +2 xuất hiện trong 759/767 đoạn EI. Cặp A ở −2 và G ở −1 xuất hiện trong 764/768 đoạn IE. Hai mẫu nằm ở hai phía khác nhau của tâm, tương ứng với phía đầu intron ở EI và phía cuối intron ở IE.

Cùng mẫu A ở −2 và G ở −1 cũng có ở 389/767 đoạn EI và 127/1.655 đoạn đối chứng. Vì vậy, khi diễn giải cần đọc cả vị trí lẫn support trong từng nhóm. Các tỷ lệ mô tả bộ UCI đã gán nhãn; đánh giá khả năng dự báo trên các chuỗi khác cần một tập kiểm chứng độc lập.